In [1]:
import h5py
from PIL import Image
import numpy as np
import pandas as pd
import os
import sys
import io
import zarr
from numcodecs import VLenUTF8
from tqdm import tqdm

this_path = os.path.abspath(os.getcwd())
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)
output_dir = os.path.join(this_path, 'datasets','zarr')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
dataset_name = 'iam' #icdar

In [4]:
input_filenames=[f'{dataset_name}_{split}_df_w_patches.csv' for split in ['train', 'public', 'private']] 

train_dfs = []
for input_filename in input_filenames:
    train_df = pd.read_csv(os.path.join(this_path, 'datasets', input_filename))
    train_df_pages = train_df.groupby('file_name', as_index=False).first()
    print(f"Processing {input_filename}...")
    print(f"Number of unique files: {len(train_df_pages)}")
    print(f"Number of unique writers: {len(train_df['writer'].unique())}")
    train_dfs.append(train_df_pages.copy())

Processing iam_train_df_w_patches.csv...
Number of unique files: 7882
Number of unique writers: 129
Processing iam_public_df_w_patches.csv...
Number of unique files: 1968
Number of unique writers: 34
Processing iam_private_df_w_patches.csv...
Number of unique files: 3099
Number of unique writers: 53


In [5]:
for input_filename, train_df_pages in zip(input_filenames, train_dfs):
    image_sizes = train_df_pages['file_name'].apply(lambda fn: Image.open(fn).size)
    train_df_pages['width'] = image_sizes.apply(lambda x: x[0])
    train_df_pages['height'] = image_sizes.apply(lambda x: x[1])
    train_df_pages['image_size'] = image_sizes
    train_df_pages[['width','height']].describe()
    max_height_idx = train_df_pages['height'].idxmax()
    max_height_file = train_df_pages.loc[max_height_idx, 'file_name']
    print("File with max height:", max_height_file)
    max_width = train_df_pages['width'].max()
    max_height = train_df_pages['height'].max()
    print("Max width:", max_width)
    print("Max height:", max_height)

    min_width = train_df_pages['width'].min()
    min_height = train_df_pages['height'].min()
    print("Min width:", min_width)
    print("Min height:", min_height)

    zarr_path = os.path.join(output_dir, f'{input_filename.replace(".csv", "")}.zarr')
    save_images_to_zarr_with_padding(train_df_pages, zarr_path, target_shape=(max_height, max_width,3))

File with max height: C:\Users\andre\PhD\Datasets\iam online\lineImages-all\lineImages\j01\j01-048\j01-048z-04.tif
Max width: 2245
Max height: 788
Min width: 77
Min height: 43


100%|██████████| 7882/7882 [04:08<00:00, 31.67it/s]


File with max height: C:\Users\andre\PhD\Datasets\iam online\lineImages-all\lineImages\a02\a02-171\a02-171z-08.tif
Max width: 2272
Max height: 628
Min width: 197
Min height: 70


100%|██████████| 1968/1968 [01:17<00:00, 25.45it/s]


File with max height: C:\Users\andre\PhD\Datasets\iam online\lineImages-all\lineImages\g06\g06-179\g06-179z-03.tif
Max width: 2233
Max height: 1377
Min width: 253
Min height: 77


100%|██████████| 3099/3099 [01:49<00:00, 28.20it/s]


# easy access

In [2]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO = reload_modules()

## functions

In [3]:
def save_images_to_hdf5(df, hdf5_path, resize=None):
    """
    df: DataFrame with 'file_name' column
    resize: tuple (width, height) to resize images, or None to keep original size
    """
    with h5py.File(hdf5_path, 'w') as f:
        num_images = len(df)
        
        # Open first image to get shape
        img = Image.open(df.iloc[0]['file_name']).convert('RGB')
        if resize:
            img = img.resize(resize)
        else:
            resize=img.size
        img_np = np.array(img)
        img_shape = img_np.shape  # (H, W, C)
        
        # Create dataset to hold all images
        dset = f.create_dataset(
            'images', shape=(num_images, *img_shape),
            dtype=img_np.dtype,
            compression="gzip",
            chunks=(1, *img_shape)  # one image per chunk
        )
        
        dt = h5py.special_dtype(vlen=str)
        f.create_dataset('filenames', (num_images,), dtype=dt)
        
        for idx, row in df.iterrows():
            img = Image.open(row['file_name']).convert('RGB')
            if resize:
                img = img.resize(resize)
            dset[idx] = np.array(img)
            f['filenames'][idx] = row['file_name']
            
            if idx % 100 == 0:
                print(f"Saved {idx+1}/{num_images} images")
    print("HDF5 file with images saved.")
def save_images_variable_length(df, hdf5_path):
    with h5py.File(hdf5_path, 'w') as f:
        num_images = len(df)
        
        # Alternative: use object dtype
        dset = f.create_dataset('images', (num_images,), dtype=h5py.special_dtype(vlen=np.dtype('uint8')))
        
        dt_str = h5py.special_dtype(vlen=str)
        f.create_dataset('filenames', (num_images,), dtype=dt_str)
        
        for idx, row in df.iterrows():
            img = Image.open(row['file_name']).convert('RGB')
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            # Convert to numpy array of uint8
            img_bytes = np.frombuffer(buf.getvalue(), dtype=np.uint8)
            dset[idx] = img_bytes
            f['filenames'][idx] = row['file_name']
            
            if idx % 100 == 0:
                print(f"Saved {idx+1}/{num_images} images")
def save_images_to_zarr(df, zarr_path, resize=None):
    num_images = len(df)

    # Open first image to get shape
    img = Image.open(df.iloc[0]['file_name']).convert('RGB')
    if resize:
        img = img.resize(resize)
    else:
        resize = img.size
    img_np = np.array(img)
    img_shape = img_np.shape  # (H, W, C)

    # Create Zarr array
    store = zarr.DirectoryStore(zarr_path)
    root = zarr.group(store=store, overwrite=True)
    zarr_array = root.create_dataset(
        'images',
        shape=(num_images, *img_shape),
        chunks=(1, *img_shape),
        dtype=img_np.dtype,
        compressor=zarr.Blosc(cname='zstd', clevel=3, shuffle=1)
    )
    filenames = root.create_dataset('filenames', shape=(num_images,), dtype=object,object_codec=VLenUTF8() ) # required for variable-length strings)

    for idx, row in tqdm(df.iterrows(), total=num_images):
        img = Image.open(row['file_name']).convert('RGB')
        if resize:
            img = img.resize(resize)
        zarr_array[idx] = np.array(img)
        filenames[idx] = row['file_name']
def pad_image_to_shape(img_np, target_shape):
    """Pad an image numpy array to the target shape (H, W, C)."""
    h, w, c = img_np.shape
    #target_h, target_w, target_c = target_shape
    padded = np.zeros(target_shape, dtype=img_np.dtype)
    padded[:h, :w, :c] = img_np
    return padded
def save_images_to_zarr_with_padding(df, zarr_path,target_shape=None):
    # Step 2: Create Zarr array
    num_images = len(df)
    store = zarr.DirectoryStore(zarr_path)
    root = zarr.group(store=store, overwrite=True)
    zarr_array = root.create_dataset(
        'images',
        shape=(num_images, *target_shape),
        chunks=(1, *target_shape),
        dtype='uint8',
        compressor=zarr.Blosc(cname='zstd', clevel=3, shuffle=1)
    )
    filenames = root.create_dataset(
        'filenames',
        shape=(num_images,),
        dtype=object,
        object_codec=VLenUTF8()
    )

    # Step 3: Read, pad, and save images
    for idx, row in tqdm(df.iterrows(), total=num_images):
        img = Image.open(row['file_name']).convert('RGB')
        img_np = np.array(img)
        padded_img = pad_image_to_shape(img_np, target_shape)
        zarr_array[idx] = padded_img
        filenames[idx] = row['file_name']